# Phase 2 holdout experiments on Kaggle

Notebook này chạy cùng một protocol cho tất cả tổ hợp:

- Holdout: leave-one-station-out (LOSO), leave-one-month-out (LOMO) và two-way unseen station + month.
- Representation: EEM nguyên bản đã flatten, `EEMpca` với 30 PCA components, và `EEMpca_SS_EC` (30 PCA scores nối thêm SS và EC). Các cột zero trong từng tập train được loại trước khi đưa vào model để giảm RAM; EEM không bị PCA. SS/EC được impute và standardize bằng thống kê của tập train trong từng fold.
- Models: Linear Regression, RBF-SVR và XGBoost.
- Mỗi outer fold lưu cả validation và test performance của **tất cả** model/representation. Notebook không chọn winner và không dùng test để chọn model; các tổ hợp được so sánh trực tiếp.
- Preprocessing (zero mask, standardization, PCA) được fit lại trên training rows của từng fold. Vì cần lưu cả validation và test, mỗi outer holdout vẫn có train/validation/test; validation chỉ là báo cáo độc lập, không dùng để chọn model.

Hãy bật **Internet** trong Kaggle. Notebook hỗ trợ public Google Drive link tới file ZIP hoặc folder chứa thư mục processed (có `eem.npy` và `samples.parquet`).


In [ ]:
# Configuration: replace DRIVE_URL with a public Google Drive ZIP link.
from pathlib import Path

DRIVE_URL = "PASTE_PUBLIC_GOOGLE_DRIVE_ZIP_LINK_HERE"
LOCAL_DATA_DIR = None  # Optional: e.g. "/kaggle/input/my-processed-data/processed"

WORK_DIR = Path("/kaggle/working")
DATA_DIR = WORK_DIR / "processed"
RESULT_DIR = WORK_DIR / "phase2_holdout_results"
ARCHIVE_PATH = WORK_DIR / "processed_data.zip"

SEED = 42
VAL_SIZE = 0.20
OUTER_TEST_SIZE = 0.20
PCA_COMPONENTS = 30
MASK_GLOBAL_ZERO_COLUMNS = True
N_JOBS = 1
SAVE_VALIDATION_METRICS = True  # Keep validation and test outputs for every candidate.
SELECT_MODEL = False  # Deliberately disabled in this notebook.

TARGETS = ["BOD", "COD", "TOC", "BOD_COD"]
REPRESENTATIONS = ["EEM", "EEMpca", "EEMpca_SS_EC"]
MODELS = ["linear", "svr", "xgboost"]
PROTOCOLS = ["station_loso", "month_lomo", "two_way"]

# Conservative CPU settings. Increase n_estimators only after a smoke run succeeds.
XGB_PARAMS = {
    "n_estimators": 150,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "verbosity": 0,
}

# Set this to an integer such as 3 for a quick smoke run. Keep None for all folds.
MAX_OUTER_FOLDS = None


In [ ]:
# Install/load dependencies. Kaggle normally already includes numpy, pandas and sklearn.
import importlib.util
import subprocess
import sys

packages = {
    "pyarrow": "pyarrow>=14",
    "xgboost": "xgboost>=2",
}
if not LOCAL_DATA_DIR and DRIVE_URL:
    packages["gdown"] = "gdown>=5.2"
for module, requirement in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", requirement])

import gc
import json
import os
import shutil
import time
import zipfile

os.environ.setdefault("OMP_NUM_THREADS", str(N_JOBS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(N_JOBS))
os.environ.setdefault("MKL_NUM_THREADS", str(N_JOBS))
os.environ.setdefault("NUMEXPR_NUM_THREADS", str(N_JOBS))


In [ ]:
# Download and locate the processed dataset.


def find_processed_dir(root: Path) -> Path | None:
    root = Path(root)
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for candidate in candidates:
        if (candidate / "eem.npy").exists() and (candidate / "samples.parquet").exists():
            return candidate
    return None

if LOCAL_DATA_DIR:
    DATA_DIR = Path(LOCAL_DATA_DIR)
else:
    existing = find_processed_dir(DATA_DIR)
    if existing is None:
        if not DRIVE_URL or DRIVE_URL.startswith("PASTE_"):
            raise ValueError("Set DRIVE_URL to a public Google Drive ZIP link before running this cell.")
        import gdown
        print("Downloading processed data from Google Drive...")
        if "/folders/" in DRIVE_URL:
            download_root = WORK_DIR / "gdrive_folder"
            if download_root.exists():
                shutil.rmtree(download_root)
            gdown.download_folder(DRIVE_URL, output=str(download_root), quiet=False, use_cookies=False)
            found = find_processed_dir(download_root)
        else:
            downloaded = gdown.download(DRIVE_URL, output=str(ARCHIVE_PATH), fuzzy=True, quiet=False)
            if not downloaded:
                raise RuntimeError("Google Drive download failed. Check that the link is public and points to a ZIP file.")
            extract_root = WORK_DIR / "processed_extracted"
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True)
            if not zipfile.is_zipfile(ARCHIVE_PATH):
                raise ValueError("The downloaded file is not a ZIP archive. Upload a ZIP containing eem.npy and samples.parquet.")
            with zipfile.ZipFile(ARCHIVE_PATH) as archive:
                archive.extractall(extract_root)
            found = find_processed_dir(extract_root)
        if found is None:
            raise FileNotFoundError("Could not find a directory containing eem.npy and samples.parquet in the downloaded resource.")
        DATA_DIR = found

print("Using processed data:", DATA_DIR)
print("Files:", sorted(p.name for p in DATA_DIR.iterdir()))


In [ ]:
# Load aligned data with memory mapping for the large original EEM array.
import numpy as np
import pandas as pd

BOD_COL = "BOD\n(0.0)"
COD_COL = "COD\n(0.0)"
TOC_COL = "TOC\n(0.0)"
SS_COL = "SS\n(0.0)"
EC_COL = "EC\n(0)"
TARGET_COLUMNS = {"BOD": BOD_COL, "COD": COD_COL, "TOC": TOC_COL, "BOD_COD": "BOD_COD"}

eem = np.load(DATA_DIR / "eem.npy", mmap_mode="r", allow_pickle=False)
samples = pd.read_parquet(DATA_DIR / "samples.parquet").reset_index(drop=True)
if "BOD_COD" not in samples.columns:
    denominator = pd.to_numeric(samples[COD_COL], errors="coerce").replace(0, np.nan)
    samples["BOD_COD"] = pd.to_numeric(samples[BOD_COL], errors="coerce") / denominator

if eem.ndim != 3 or len(eem) != len(samples):
    raise ValueError(f"EEM/metadata mismatch: eem={eem.shape}, samples={samples.shape}")
if not np.isfinite(np.asarray(eem[: min(len(eem), 3)])).all():
    raise ValueError("EEM contains NaN or Inf")

# The processed grid has 5,928 cells that are zero in every sample. Compute the
# mask in chunks so the full flattened tensor is never duplicated in RAM.
def compute_global_nonzero_mask(eem_array, chunk_size=64):
    n_rows = len(eem_array)
    n_features = int(np.prod(eem_array.shape[1:]))
    mask = np.zeros(n_features, dtype=bool)
    for start in range(0, n_rows, chunk_size):
        block = np.asarray(eem_array[start : start + chunk_size], dtype=np.float32).reshape(-1, n_features)
        mask |= np.any(np.abs(block) > 0, axis=0)
    return mask

GLOBAL_FEATURE_MASK = compute_global_nonzero_mask(eem) if MASK_GLOBAL_ZERO_COLUMNS else np.ones(int(np.prod(eem.shape[1:])), dtype=bool)
print({"samples": len(samples), "stations": samples["Point"].nunique(), "months": samples["Month"].nunique(), "eem_shape": tuple(eem.shape), "raw_features": len(GLOBAL_FEATURE_MASK), "kept_features": int(GLOBAL_FEATURE_MASK.sum()), "zero_features": int((~GLOBAL_FEATURE_MASK).sum())})


In [ ]:
# Feature preparation, models and metrics.
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor


def finite_target_indices(indices, y):
    indices = np.asarray(indices, dtype=int)
    return indices[np.isfinite(y[indices])]


def flatten_rows(eem_array, indices, mask):
    indices = np.asarray(indices, dtype=int)
    block = np.asarray(eem_array[indices], dtype=np.float32).reshape(len(indices), -1)
    return block[:, mask]


def tabular_rows(indices):
    """Return SS and EC as numeric values for the requested sample rows."""
    required = [SS_COL, EC_COL]
    missing = [column for column in required if column not in samples.columns]
    if missing:
        raise KeyError(f"Processed metadata is missing required tabular columns: {missing}")
    return (
        samples.iloc[np.asarray(indices, dtype=int)][required]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=float)
    )


def make_features(eem_array, train_indices, eval_indices, representation):
    """Fit masks and transforms on train_indices, then apply them to eval_indices."""
    if MASK_GLOBAL_ZERO_COLUMNS:
        # Match FeatureBuilder in the main Phase-0 pipeline: the mask is learned
        # from the training rows of this fold, never from validation/test rows.
        train_mask = compute_global_nonzero_mask(eem_array[np.asarray(train_indices, dtype=int)])
    else:
        train_mask = np.ones(int(np.prod(eem_array.shape[1:])), dtype=bool)
    x_train = flatten_rows(eem_array, train_indices, train_mask)
    x_eval = flatten_rows(eem_array, eval_indices, train_mask)
    if representation == "EEM":
        return x_train, x_eval, {
            "n_features": x_train.shape[1],
            "pca_components": 0,
            "tabular_features": 0,
            "zero_masked_features": int((~train_mask).sum()),
        }
    if representation not in {"EEMpca", "EEMpca_SS_EC"}:
        raise ValueError(representation)

    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_eval_scaled = scaler.transform(x_eval)
    n_components = min(PCA_COMPONENTS, x_train_scaled.shape[0], x_train_scaled.shape[1])
    pca = PCA(n_components=n_components, svd_solver="randomized", random_state=SEED)
    x_train_pca = pca.fit_transform(x_train_scaled).astype(np.float32, copy=False)
    x_eval_pca = pca.transform(x_eval_scaled).astype(np.float32, copy=False)

    tabular_features = 0
    if representation == "EEMpca_SS_EC":
        # Imputation and scaling are fit only on the inner/outer training rows.
        # This prevents SS/EC distribution information from leaking into val/test.
        tabular_pipeline = make_pipeline(
            SimpleImputer(strategy="median", keep_empty_features=True),
            StandardScaler(),
        )
        tab_train = tabular_pipeline.fit_transform(tabular_rows(train_indices))
        tab_eval = tabular_pipeline.transform(tabular_rows(eval_indices))
        x_train_pca = np.hstack([x_train_pca, tab_train.astype(np.float32, copy=False)])
        x_eval_pca = np.hstack([x_eval_pca, tab_eval.astype(np.float32, copy=False)])
        tabular_features = 2

    return x_train_pca, x_eval_pca, {
        "n_features": x_train_pca.shape[1],
        "pca_components": n_components,
        "tabular_features": tabular_features,
        "zero_masked_features": int((~train_mask).sum()),
    }


def make_model(name):
    if name == "linear":
        return LinearRegression(n_jobs=N_JOBS)
    if name == "svr":
        return TransformedTargetRegressor(
            regressor=make_pipeline(StandardScaler(), SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale")),
            transformer=StandardScaler(),
        )
    if name == "xgboost":
        from xgboost import XGBRegressor
        return XGBRegressor(random_state=SEED, n_jobs=N_JOBS, **XGB_PARAMS)
    raise ValueError(name)


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    result = {
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) >= 2 and np.unique(y_true).size > 1 else np.nan,
        "MSE": float(mean_squared_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE": float(mean_absolute_percentage_error(y_true, y_pred)),
        "n": int(len(y_true)),
    }
    return result


def baseline_predictions(fit_indices, eval_indices, y):
    global_mean = float(np.mean(y[fit_indices]))
    global_pred = np.full(len(eval_indices), global_mean, dtype=float)
    if "Point" not in samples.columns:
        return {"global_mean": global_pred, "station_mean": global_pred.copy()}
    fit_frame = pd.DataFrame({"Point": samples.iloc[fit_indices]["Point"].to_numpy(), "target": y[fit_indices]})
    station_means = fit_frame.groupby("Point")["target"].mean()
    station_pred = pd.Series(samples.iloc[eval_indices]["Point"].to_numpy()).map(station_means).fillna(global_mean).to_numpy(dtype=float)
    return {"global_mean": global_pred, "station_mean": station_pred}


In [ ]:
# Outer holdout definitions.
from sklearn.model_selection import GroupShuffleSplit

ALL_INDICES = np.arange(len(samples), dtype=int)


def choose_held_groups(values, test_size, seed):
    values = pd.Series(values).dropna().drop_duplicates().to_numpy()
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    _, held_rel = next(splitter.split(values, groups=values))
    return values[held_rel]


def make_loso_specs(group_col, max_folds=None):
    values = samples[group_col]
    groups = sorted(values.dropna().unique().tolist(), key=str)
    if max_folds is not None:
        groups = groups[:max_folds]
    specs = []
    for fold, group in enumerate(groups, 1):
        test_mask = values.eq(group).to_numpy()
        outer_train = ALL_INDICES[~test_mask]
        splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED + fold)
        train_rel, val_rel = next(splitter.split(outer_train, groups=values.iloc[outer_train].to_numpy()))
        train = outer_train[train_rel]
        validation = outer_train[val_rel]
        specs.append({"fold": fold, "held_out_group": str(group), "train": train, "validation": validation, "outer_train": outer_train, "test": ALL_INDICES[test_mask], "excluded": np.array([], dtype=int)})
    return specs


def make_two_way_spec():
    held_station = choose_held_groups(samples["Point"], OUTER_TEST_SIZE, SEED)
    held_month = choose_held_groups(samples["Month"], OUTER_TEST_SIZE, SEED + 1)
    station = samples["Point"].to_numpy()
    month = samples["Month"].to_numpy()
    test_mask = np.isin(station, held_station) & np.isin(month, held_month)
    eligible_mask = ~np.isin(station, held_station) & ~np.isin(month, held_month)
    test = ALL_INDICES[test_mask]
    eligible = ALL_INDICES[eligible_mask]
    splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
    train_rel, val_rel = next(splitter.split(eligible, groups=station[eligible]))
    train = eligible[train_rel]
    validation = eligible[val_rel]
    excluded = np.setdiff1d(ALL_INDICES, np.concatenate([train, validation, test]))
    return [{"fold": 1, "held_out_group": f"Point={list(held_station)};Month={list(held_month)}", "train": train, "validation": validation, "outer_train": np.concatenate([train, validation]), "test": test, "excluded": excluded}]


def make_specs(protocol):
    if protocol == "station_loso":
        return make_loso_specs("Point", MAX_OUTER_FOLDS)
    if protocol == "month_lomo":
        return make_loso_specs("Month", MAX_OUTER_FOLDS)
    if protocol == "two_way":
        return make_two_way_spec()
    raise ValueError(protocol)


In [ ]:
# Evaluation loop. Results are written after every protocol so a long Kaggle run is resumable.
def evaluate_protocol(protocol):
    protocol_dir = RESULT_DIR / protocol
    protocol_dir.mkdir(parents=True, exist_ok=True)
    specs = make_specs(protocol)
    metrics_rows = []
    prediction_rows = []
    started = time.time()

    for target_name in TARGETS:
        target_col = TARGET_COLUMNS[target_name]
        y = pd.to_numeric(samples[target_col], errors="coerce").to_numpy(dtype=float)
        for spec in specs:
            train = finite_target_indices(spec["train"], y)
            validation = finite_target_indices(spec["validation"], y)
            outer_train = finite_target_indices(spec["outer_train"], y)
            test = finite_target_indices(spec["test"], y)
            if min(len(train), len(validation), len(outer_train), len(test)) < 2:
                print(f"Skipping {protocol}/{target_name}/fold={spec['fold']}: insufficient finite targets")
                continue

            fold_common = {
                "protocol": protocol,
                "fold": spec["fold"],
                "held_out_group": spec["held_out_group"],
                "target": target_name,
                "n_train_for_validation": len(train),
                "n_validation": len(validation),
                "n_outer_train_for_test": len(outer_train),
                "n_test": len(test),
                "n_excluded": len(spec["excluded"]),
            }

            # Baselines are written for both partitions and are not candidates.
            val_baselines = baseline_predictions(train, validation, y)
            test_baselines = baseline_predictions(outer_train, test, y)
            for baseline_name in ["global_mean", "station_mean"]:
                for split_name, indices, pred in [
                    ("validation", validation, val_baselines[baseline_name]),
                    ("test", test, test_baselines[baseline_name]),
                ]:
                    metric = regression_metrics(y[indices], pred)
                    metrics_rows.append({
                        **fold_common,
                        "representation": "baseline",
                        "model": baseline_name,
                        "split": split_name,
                        **metric,
                    })
                    prediction_rows.extend(
                        {
                            **fold_common,
                            "representation": "baseline",
                            "model": baseline_name,
                            "split": split_name,
                            "row_position": int(row),
                            "y_true": float(y[row]),
                            "y_pred": float(value),
                        }
                        for row, value in zip(indices, pred)
                    )

            for representation in REPRESENTATIONS:
                # Fit preprocessing on train for validation, then refit on all
                # outer-train rows for the untouched outer test.
                x_train, x_val, feature_info = make_features(eem, train, validation, representation)
                x_outer_train, x_test, _ = make_features(eem, outer_train, test, representation)
                for model_name in MODELS:
                    # Every requested model/representation is evaluated. There is
                    # intentionally no validation-based winner selection here.
                    model = make_model(model_name)
                    model.fit(x_train, y[train])
                    val_pred = np.asarray(model.predict(x_val), dtype=float)
                    val_metric = regression_metrics(y[validation], val_pred)
                    metrics_rows.append({
                        **fold_common,
                        "representation": representation,
                        "model": model_name,
                        "split": "validation",
                        **feature_info,
                        **val_metric,
                    })
                    prediction_rows.extend(
                        {
                            **fold_common,
                            "representation": representation,
                            "model": model_name,
                            "split": "validation",
                            "row_position": int(row),
                            "y_true": float(y[row]),
                            "y_pred": float(value),
                        }
                        for row, value in zip(validation, val_pred)
                    )

                    # Fit the same candidate on all outer-train rows and evaluate
                    # the untouched test rows. Every candidate gets its own test
                    # metrics; no candidate is selected.
                    refit_model = make_model(model_name)
                    refit_model.fit(x_outer_train, y[outer_train])
                    test_pred = np.asarray(refit_model.predict(x_test), dtype=float)
                    test_metric = regression_metrics(y[test], test_pred)
                    metrics_rows.append({
                        **fold_common,
                        "representation": representation,
                        "model": model_name,
                        "split": "test",
                        **feature_info,
                        **test_metric,
                    })
                    prediction_rows.extend(
                        {
                            **fold_common,
                            "representation": representation,
                            "model": model_name,
                            "split": "test",
                            "row_position": int(row),
                            "y_true": float(y[row]),
                            "y_pred": float(value),
                        }
                        for row, value in zip(test, test_pred)
                    )
                    del model, refit_model, val_pred, test_pred
                    gc.collect()
                del x_train, x_val, x_outer_train, x_test
                gc.collect()

            print(
                f"{protocol} | {target_name} | fold {spec['fold']}/{len(specs)} | "
                f"evaluated {len(REPRESENTATIONS)} representations x {len(MODELS)} models",
                flush=True,
            )

            # Checkpoint after each fold.
            pd.DataFrame(metrics_rows).to_csv(protocol_dir / "metrics_long_partial.csv", index=False)
            pd.DataFrame(prediction_rows).to_csv(protocol_dir / "predictions_partial.csv", index=False)

    metrics = pd.DataFrame(metrics_rows)
    predictions = pd.DataFrame(prediction_rows)
    metrics.to_csv(protocol_dir / "metrics_long.csv", index=False)
    predictions.to_csv(protocol_dir / "predictions.csv", index=False)
    print(f"Finished {protocol}: {len(metrics):,} metric rows in {(time.time()-started)/60:.1f} min")
    return metrics, predictions


def summarize(metrics, predictions):
    summary_rows = []
    group_cols = ["protocol", "target", "representation", "model", "split"]
    for key, group in metrics.groupby(group_cols, dropna=False):
        pred = predictions
        for col, value in zip(group_cols, key):
            pred = pred[pred[col].eq(value)]
        y_true = pred["y_true"].to_numpy(dtype=float)
        y_pred = pred["y_pred"].to_numpy(dtype=float)
        pooled = regression_metrics(y_true, y_pred) if len(pred) >= 2 else {"R2": np.nan, "RMSE": np.nan, "MAE": np.nan, "MSE": np.nan, "MAPE": np.nan, "n": len(pred)}
        summary_rows.append({
            **dict(zip(group_cols, key)),
            "n_folds": int(group["fold"].nunique()),
            "n_samples": int(len(pred)),
            "R2_mean": float(group["R2"].mean()),
            "R2_std": float(group["R2"].std(ddof=1)) if len(group) > 1 else 0.0,
            "R2_pooled": pooled["R2"],
            "RMSE_mean": float(group["RMSE"].mean()),
            "RMSE_pooled": pooled["RMSE"],
            "MAE_mean": float(group["MAE"].mean()),
            "MAE_pooled": pooled["MAE"],
        })
    return pd.DataFrame(summary_rows)


In [ ]:
# Run all requested protocols. For a first smoke test, set MAX_OUTER_FOLDS=2,
# MODELS=["linear"], and/or REPRESENTATIONS=["EEMpca_SS_EC"] in the configuration cell.
if RESULT_DIR.exists():
    print("Existing result directory found; new files will overwrite protocol outputs.")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
all_metrics, all_predictions = [], []
for protocol in PROTOCOLS:
    metrics, predictions = evaluate_protocol(protocol)
    all_metrics.append(metrics)
    all_predictions.append(predictions)

all_metrics = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()
all_predictions = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
all_summary = summarize(all_metrics, all_predictions)

all_metrics.to_csv(RESULT_DIR / "holdout_metrics_long.csv", index=False)
all_predictions.to_csv(RESULT_DIR / "holdout_predictions.csv", index=False)
all_summary.to_csv(RESULT_DIR / "holdout_metrics_summary.csv", index=False)
with open(RESULT_DIR / "run_config.json", "w", encoding="utf-8") as handle:
    json.dump({
        "seed": SEED,
        "val_size": VAL_SIZE,
        "outer_test_size": OUTER_TEST_SIZE,
        "pca_components": PCA_COMPONENTS,
        "mask_global_zero_columns": MASK_GLOBAL_ZERO_COLUMNS,
        "raw_eem_shape": list(eem.shape),
        "raw_features": int(len(GLOBAL_FEATURE_MASK)),
        "kept_features_global_summary": int(GLOBAL_FEATURE_MASK.sum()),
        "zero_mask_fit": "training rows separately for each fold and representation",
        "targets": TARGETS,
        "representations": REPRESENTATIONS,
        "tabular_augmented_representation": {
            "name": "EEMpca_SS_EC",
            "columns": [SS_COL, EC_COL],
            "preprocessing": "median imputation + standardization fit separately on each training fold",
        },
        "models": MODELS,
        "protocols": PROTOCOLS,
        "model_selection": "none",
        "validation_saved": SAVE_VALIDATION_METRICS,
        "xgb_params": XGB_PARAMS,
        "data_dir": str(DATA_DIR),
    }, handle, indent=2)

print("Saved:", RESULT_DIR)
print(all_summary.sort_values(["protocol", "target", "split", "representation", "model"]).head(30).to_string(index=False))


In [ ]:
# Package the complete result directory for download from Kaggle.
archive = shutil.make_archive(str(RESULT_DIR.parent / "phase2_holdout_results"), "zip", RESULT_DIR)
print("Download this file:", archive)
print("Main files:")
for path in sorted(RESULT_DIR.glob("*.csv")):
    print(" -", path.name, path.stat().st_size / 1024**2, "MB")
